### <p style="text-align: center;"> Análisis exploratorio de datos (EDA) con Spark para información de ventas 2023</p>

#### <p style="text-align: center;"> Tarea 2 – Análisis de Datos, Tendencias Tecnológicas y metodologías.</p>

<p style="text-align: center;"> Nicole Fontecha Lázaro<br>1101760540</p>

<p style="text-align: center;"> Minería para Big Data<br>Grupo 2</p>   

<p style="text-align: center;"> UNIVERSIDAD NACIONAL ABIERTA Y A DISTANCIA - UNAD </p>

<p style="text-align: center;"> Bogotá, March 17th </p>

### Importación de librerías

In [74]:
import findspark 
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pyspark.sql.functions as F
import matplotlib.ticker as mticker
from pyspark.sql import SparkSession
from pyspark.sql.functions import   col, avg, sum, count, month, year, dayofweek, quarter, \
                                    max, min, stddev, corr, when, round, to_date

### Conexión a Spark

In [75]:
findspark.init() #Ubicación automática de Spark

spark = SparkSession.builder \
    .appName("EDA_Ventas2023_UNAD") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate() #Crear conexión con Spark

spark.sparkContext.setLogLevel("ERROR")

print("Successfully connected to Spark")


Successfully connected to Spark


### Data loading

In [76]:
df = spark.read.csv(                #Carga/Lee el CSV
    "../resources/ventas_2023.csv", #Ruta del archivo
    header=True,                    #Ubica la primera fila como encabezado con el nombre de las columnas
    inferSchema=True                #DEtecta automáticamente los tipos de dato
) 


df.show(3) #Mostrar los primero 3 registros para validar la estructura

+----------+-------+----------+------+---------+--------+---+----------+
|     fecha|  monto|  producto|region|tipo_pago|cantidad|mes|dia_semana|
+----------+-------+----------+------+---------+--------+---+----------+
|2023-01-01| 786.92|Smartphone|  Este|  Crédito|       5|  1|   Domingo|
|2023-01-02|1211.86|Smartwatch| Oeste|   Débito|      10|  1|     Lunes|
|2023-01-03|1072.96|Smartphone|  Este|  Crédito|       6|  1|    Martes|
+----------+-------+----------+------+---------+--------+---+----------+
only showing top 3 rows


In [77]:
print("Esquema de la data") 
df.printSchema()

print(f"\nDimensions: {df.count()} filas × {len(df.columns)} columnas")
print("Columns:",df.columns)

print("\nEstadísticas descriptivas básicas:") 
df.describe().show()

Esquema de la data
root
 |-- fecha: date (nullable = true)
 |-- monto: double (nullable = true)
 |-- producto: string (nullable = true)
 |-- region: string (nullable = true)
 |-- tipo_pago: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- dia_semana: string (nullable = true)


Dimensions: 365 filas × 8 columnas
Columns: ['fecha', 'monto', 'producto', 'region', 'tipo_pago', 'cantidad', 'mes', 'dia_semana']

Estadísticas descriptivas básicas:
+-------+-----------------+--------+------+---------+-----------------+-----------------+----------+
|summary|            monto|producto|region|tipo_pago|         cantidad|              mes|dia_semana|
+-------+-----------------+--------+------+---------+-----------------+-----------------+----------+
|  count|              365|     365|   365|      365|              365|              365|       365|
|   mean|993.5168493150684|    NULL|  NULL|     NULL|5.432876712328767|6.526027397260274|    

### Análisis exploratorio de datos (EDA – Exploratory Data Analysis)

##### Identificación de ruido

In [78]:
print("Valores nulos por columna:")
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

total      = df.count()
sin_duplic = df.dropDuplicates().count()    #Conteo de registros únicos
duplicados = total - sin_duplic             #Cálculo de registros duplicados
print(f"\nRegistros duplicados: {duplicados} ({duplicados/total*100:.2f}%)")

invalidos = df.filter(col("monto") < 0).count() #Cálculo de valores inválidos (monto negativo)
print(f"\nRegistros con monto < 0: {invalidos}")

print("\nRango de tiempo analizado:")
df.select(
    min("fecha").alias("fecha_inicio"),
    max("fecha").alias("fecha_fin")
).show()

print("Valores únicos por variable categórica")
for c in ["producto", "region", "tipo_pago", "dia_semana"]:
    vals = [r[c] for r in df.select(c).distinct().collect()]
    print(f"{c}: {vals}")


Valores nulos por columna:
+-----+-----+--------+------+---------+--------+---+----------+
|fecha|monto|producto|region|tipo_pago|cantidad|mes|dia_semana|
+-----+-----+--------+------+---------+--------+---+----------+
|    0|    0|       0|     0|        0|       0|  0|         0|
+-----+-----+--------+------+---------+--------+---+----------+


Registros duplicados: 0 (0.00%)

Registros con monto < 0: 0

Rango de tiempo analizado:
+------------+----------+
|fecha_inicio| fecha_fin|
+------------+----------+
|  2023-01-01|2023-12-31|
+------------+----------+

Valores únicos por variable categórica
producto: ['Smartwatch', 'Smartphone', 'Laptop', 'Tablet']
region: ['Oeste', 'Sur', 'Este', 'Norte']
tipo_pago: ['Crédito', 'Efectivo', 'Débito']
dia_semana: ['Domingo', 'Miércoles', 'Jueves', 'Sábado', 'Martes', 'Viernes', 'Lunes']


In [79]:
spark.stop()
print("Sesión de Spark cerrada correctamente")


Sesión de Spark cerrada correctamente
